In [ ]:
import os
os.environ["QT_PLUGIN_PATH"] = ""
os.environ["QT_QPA_PLATFORM_PLUGIN_PATH"] = ""
from PyQt5.QtCore import QLibraryInfo, QLibraryInfo
print("Qt path:", QLibraryInfo.location(QLibraryInfo.PrefixPath))

In [ ]:
import sys,os
sys.path.append('../../package/')
%run ../../package/init_jupyter.py

In [ ]:
%gui qt
import sys
from PyQt5.QtWidgets import QApplication,QStyleFactory
print(QStyleFactory.keys())

app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

try:
    app.setStyle('Fusion')
except Exception as e:
    print(f"Failed to set style: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mujoco as mj
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import clear_output, display
from scipy.spatial import distance, procrustes
import json
sys.path.append('../')
sys.path.append('../../asset/')
sys.path.append('../../package/') # for 'preretarget'
sys.path.append('../../package/bvh_helper/') # for 'bvh_parser'
sys.path.append('../../package/yoon/')
sys.path.append('../../package/mujoco_helper/')
sys.path.append('../../package/motion_retarget')
sys.path.append('../../package/kinematics_helper/')
sys.path.append('../../package/utility/')
sys.path.append('../../package/smpl/')
sys.path.append('../../package/irvae_tools/')
from bvh_parser import * # for cmu mocap parsing
from motion_retarget import *
from geometry import relaxed_distortion_measure
from clustering_robots import *
from _robot_info import *
from mujoco_parser import *
from utils import *
from smpl_model import *
from smpl_parser import *

plt.rc('xtick', labelsize=6)
plt.rc('ytick', labelsize=6)
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
print ("PyTorch:[%s]"%(th.__version__))
print("MuJoCo:[%s]" % mj.__version__)

### Load Robot Model

In [ ]:
# Load Humanoid
xml_path = '../../asset/lego_middle/scene_lego_middle.xml'
env = MuJoCoParserClass(name='',rel_xml_path=xml_path,verbose=False)

In [ ]:
bvh_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
secs,chains = get_chains_from_bvh_cmu_mocap(
    bvh_path         = bvh_path,
    p_rate           = 0.056444*1.0,
    zup              = True,
    R_offset         = rpy2r(np.deg2rad([0,0,90])),
    plot_chain_graph = True,
    verbose          = False,
)
secs,chains = secs[1:],chains[1:] # exlude the first frame (T-pose)
L = len(chains); dt = secs[1]-secs[0]; HZ = int(1/dt)
print ("[%s] loaded. L:[%d] dt:[%.4f]sec HZ:[%d]"%(bvh_path,L,dt,HZ))

In [ ]:
# joint_names_for_ik      = [
#     'torso_yaw_joint','torso_roll_joint',
#     'l_shoulder_pitch_joint','l_shoulder_roll_joint','l_shoulder_yaw_joint',
#     'l_elbow_pitch_joint','l_elbow_yaw_joint',
#     'l_wrist_roll_joint','l_wrist_yaw_joint',

#     'r_shoulder_pitch_joint','r_shoulder_roll_joint','r_shoulder_yaw_joint',
#     'r_elbow_pitch_joint','r_elbow_yaw_joint',
#     'r_wrist_roll_joint','r_wrist_yaw_joint',

#     'r_pelvis_pitch_joint','r_knee_pitch_joint','r_ankle_pitch_joint',
#     'l_pelvis_pitch_joint','l_knee_pitch_joint','l_ankle_pitch_joint']

qpos_dir_list = directional_preretarget_postrigging(
    env,
    secs,
    chains,
    # ik_p_spine=False,
    # ik_p_neck=ik_p_neck,
    # leg_fixed=leg_fixed,
    src_name           = 'cmu',
    trgt_name          = 'lego_middle',
    joint_names_for_ik = None,
    max_ik_tick        = 100, # maximum IK iteration
    ik_stepsize        = 1, # IK stepsize
    ik_err_th          = 1e-3, # IK error threshold
    print_ik_failure   = False,
    ik_R_hand          = False,
    ik_R_foot          = False,
    reset_env          = True,
    animate            = False, # animate pre-retarget process
    plot_joi_src       = False,
    plot_joi_trgt      = False,
    render_every       = 1,
    verbose            = True,
    print_every        = 30,
    pause_each_tick    = False,
    post_rigging       = True,
    override           = None,
)

In [ ]:
qpos_dir_list[485]

In [ ]:
import mediapy
env.init_viewer(title='Pre-Retargeting',width=800,height=800)
env.set_viewer(azimuth=169.57500000000022,distance=3.4430763972682676,elevation=-12.974999999999993,lookat=[ 0.02, -0.03,  0.9 ])
# env.set_viewer(azimuth=150.4500000000003,distance=5.925655229922415,elevation=-20.849999999999973,lookat=[ 0.9 ,  1.66, -0.81])
env.reset()
SAVE_VIDEO = True
video = []
tick = 0

env.viewer_pause()
# env.plot_traj(traj=base_dir_traj,rgba=(0,0,0,1),
#                 plot_line=False,plot_cylinder=True,cylinder_r=0.005)
# env.plot_traj(traj=rf_dir_traj,rgba=(1,0,0,1),
#                 plot_line=False,plot_cylinder=True,cylinder_r=0.005)
# env.plot_traj(traj=lf_dir_traj,rgba=(0,0,1,1),
#                 plot_line=False,plot_cylinder=True,cylinder_r=0.005)
while tick < L and env.is_viewer_alive():
# while tick < L//2 and env.is_viewer_alive():
    q_robot = qpos_dir_list[tick]
    env.forward(q=q_robot)
    # env.plot_T(p=np.zeros(3),R=np.eye(3),label="Tick: [%d/%d]"%(tick,L),axis_len=0.5,axis_width=0.005)
#     env.plot_contact_info(
#             r_arrow             = 0.008,
#             h_arrow             = 0.5,
#     )
    env.render()
    video.append(env.grab_image())
    tick += 1
    if tick == L:
        if not SAVE_VIDEO: tick = 0
env.close_viewer()
if SAVE_VIDEO:mediapy.show_video(images=video,fps=60,width=800)